In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

BRONZE_SCHEMA = "supplysage_bronze"

# Use the same batch id from Notebook 05.
INGESTION_BATCH_ID = "ext_api_20260609_225257_ff403938"

VALIDATION_RUN_ID = f"external_validation_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "06_bronze_validate_external_risk_sources"

print("VALIDATION_RUN_ID:", VALIDATION_RUN_ID)
print("INGESTION_BATCH_ID:", INGESTION_BATCH_ID)
print("BRONZE_SCHEMA:", BRONZE_SCHEMA)

In [0]:
expected_external_tables = [
    {
        "source_name": "gdelt",
        "table_name": "bronze_ext_gdelt_doc_raw",
        "expected_min_rows": 1,
        "batch_filtered": True
    },
    {
        "source_name": "nws",
        "table_name": "bronze_ext_nws_alerts_raw",
        "expected_min_rows": 3,
        "batch_filtered": True
    },
    {
        "source_name": "cisa",
        "table_name": "bronze_ext_cisa_kev_raw",
        "expected_min_rows": 1,
        "batch_filtered": True
    },
    {
        "source_name": "cpsc",
        "table_name": "bronze_ext_cpsc_recalls_raw",
        "expected_min_rows": 1,
        "batch_filtered": True
    },
    {
        "source_name": "ofac",
        "table_name": "bronze_ext_ofac_sanctions_raw",
        "expected_min_rows": 2,
        "batch_filtered": True
    },
    {
        "source_name": "sec_edgar",
        "table_name": "bronze_ext_sec_company_submissions_raw",
        "expected_min_rows": 6,
        "batch_filtered": True
    },
    {
        "source_name": "eia",
        "table_name": "bronze_ext_eia_fuel_prices_raw",
        "expected_min_rows": 1,
        "batch_filtered": True
    },
    {
        "source_name": "source_registry",
        "table_name": "bronze_ext_source_registry",
        "expected_min_rows": 7,
        "batch_filtered": False
    },
    {
        "source_name": "api_runs",
        "table_name": "bronze_ext_api_runs",
        "expected_min_rows": 15,
        "batch_filtered": True
    }
]


def bronze_table_exists(table_name):
    try:
        spark.sql(f"DESCRIBE TABLE {BRONZE_SCHEMA}.{table_name}")
        return True
    except Exception:
        return False


validation_rows = []

for item in expected_external_tables:
    source_name = item["source_name"]
    table_name = item["table_name"]
    expected_min_rows = item["expected_min_rows"]
    batch_filtered = item["batch_filtered"]

    full_table_name = f"{BRONZE_SCHEMA}.{table_name}"
    exists_flag = bronze_table_exists(table_name)

    if exists_flag:
        if batch_filtered:
            actual_row_count = (
                spark.table(full_table_name)
                .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
                .count()
            )
        else:
            actual_row_count = spark.table(full_table_name).count()

        status = "PASS" if actual_row_count >= expected_min_rows else "FAIL"
        issue_detail = None if status == "PASS" else f"Expected at least {expected_min_rows} rows, found {actual_row_count}."
    else:
        actual_row_count = 0
        status = "FAIL"
        issue_detail = "Table does not exist."

    validation_rows.append({
        "validation_run_id": VALIDATION_RUN_ID,
        "ingestion_batch_id": INGESTION_BATCH_ID,
        "source_name": source_name,
        "table_name": table_name,
        "validation_check": "table_exists_and_min_row_count",
        "expected_value": str(expected_min_rows),
        "actual_value": str(actual_row_count),
        "status": status,
        "issue_detail": issue_detail,
        "validated_at": datetime.now(timezone.utc).isoformat(),
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })


validation_schema = T.StructType([
    T.StructField("validation_run_id", T.StringType(), True),
    T.StructField("ingestion_batch_id", T.StringType(), True),
    T.StructField("source_name", T.StringType(), True),
    T.StructField("table_name", T.StringType(), True),
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True),
    T.StructField("validated_at", T.StringType(), True),
    T.StructField("created_by", T.StringType(), True),
    T.StructField("notebook_name", T.StringType(), True)
])

table_validation_df = spark.createDataFrame(validation_rows, schema=validation_schema)

display(
    table_validation_df
    .orderBy("status", "source_name")
)

In [0]:
display(
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_api_runs")
    .groupBy("ingestion_batch_id", "source_name")
    .agg(
        F.count("*").alias("api_call_count"),
        F.sum(F.when(F.col("response_ok") == True, 1).otherwise(0)).alias("successful_calls"),
        F.sum(F.when(F.col("response_ok") == False, 1).otherwise(0)).alias("failed_calls"),
        F.collect_set("response_status_code").alias("status_codes"),
        F.max("fetched_at").alias("latest_fetched_at")
    )
    .orderBy(F.col("latest_fetched_at").desc(), "ingestion_batch_id", "source_name")
)

In [0]:
external_raw_tables = [
    {
        "source_name": "gdelt",
        "table_name": "bronze_ext_gdelt_doc_raw"
    },
    {
        "source_name": "nws",
        "table_name": "bronze_ext_nws_alerts_raw"
    },
    {
        "source_name": "cisa",
        "table_name": "bronze_ext_cisa_kev_raw"
    },
    {
        "source_name": "cpsc",
        "table_name": "bronze_ext_cpsc_recalls_raw"
    },
    {
        "source_name": "ofac",
        "table_name": "bronze_ext_ofac_sanctions_raw"
    },
    {
        "source_name": "sec_edgar",
        "table_name": "bronze_ext_sec_company_submissions_raw"
    },
    {
        "source_name": "eia",
        "table_name": "bronze_ext_eia_fuel_prices_raw"
    }
]

required_raw_columns = [
    "api_run_id",
    "ingestion_batch_id",
    "source_name",
    "endpoint_name",
    "request_url",
    "query_params_json",
    "request_headers_json",
    "response_status_code",
    "response_ok",
    "fetched_at",
    "raw_payload_json",
    "payload_hash",
    "error_message",
    "ingestion_version",
    "created_by",
    "notebook_name"
]

critical_non_null_columns = [
    "api_run_id",
    "ingestion_batch_id",
    "source_name",
    "endpoint_name",
    "request_url",
    "response_status_code",
    "response_ok",
    "fetched_at",
    "raw_payload_json",
    "payload_hash"
]

column_validation_rows = []

for item in external_raw_tables:
    source_name = item["source_name"]
    table_name = item["table_name"]
    full_table_name = f"{BRONZE_SCHEMA}.{table_name}"

    df = spark.table(full_table_name).filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    existing_columns = set(df.columns)

    missing_columns = [
        col_name for col_name in required_raw_columns
        if col_name not in existing_columns
    ]

    if len(missing_columns) == 0:
        status = "PASS"
        issue_detail = None
    else:
        status = "FAIL"
        issue_detail = "Missing columns: " + ", ".join(missing_columns)

    column_validation_rows.append({
        "validation_run_id": VALIDATION_RUN_ID,
        "ingestion_batch_id": INGESTION_BATCH_ID,
        "source_name": source_name,
        "table_name": table_name,
        "validation_check": "required_metadata_columns_exist",
        "expected_value": ",".join(required_raw_columns),
        "actual_value": ",".join(df.columns),
        "status": status,
        "issue_detail": issue_detail,
        "validated_at": datetime.now(timezone.utc).isoformat(),
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })

    if len(missing_columns) == 0:
        null_condition = None

        for col_name in critical_non_null_columns:
            condition_piece = F.col(col_name).isNull()

            if null_condition is None:
                null_condition = condition_piece
            else:
                null_condition = null_condition | condition_piece

        null_row_count = df.filter(null_condition).count()

        status = "PASS" if null_row_count == 0 else "FAIL"
        issue_detail = None if status == "PASS" else f"{null_row_count} rows have nulls in critical metadata columns."

        column_validation_rows.append({
            "validation_run_id": VALIDATION_RUN_ID,
            "ingestion_batch_id": INGESTION_BATCH_ID,
            "source_name": source_name,
            "table_name": table_name,
            "validation_check": "critical_metadata_columns_not_null",
            "expected_value": "0",
            "actual_value": str(null_row_count),
            "status": status,
            "issue_detail": issue_detail,
            "validated_at": datetime.now(timezone.utc).isoformat(),
            "created_by": CREATED_BY,
            "notebook_name": NOTEBOOK_NAME
        })


column_validation_df = spark.createDataFrame(column_validation_rows, schema=validation_schema)

display(
    column_validation_df
    .orderBy("status", "source_name", "validation_check")
)

In [0]:
payload_validation_rows = []

for item in external_raw_tables:
    source_name = item["source_name"]
    table_name = item["table_name"]
    full_table_name = f"{BRONZE_SCHEMA}.{table_name}"

    df = spark.table(full_table_name).filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)

    total_rows = df.count()

    failed_response_count = (
        df
        .filter(F.col("response_ok") != True)
        .count()
    )

    empty_payload_count = (
        df
        .filter(
            F.col("raw_payload_json").isNull()
            | (F.length(F.trim(F.col("raw_payload_json"))) == 0)
            | (F.col("raw_payload_json") == "null")
        )
        .count()
    )

    duplicate_payload_hash_count = (
        df
        .groupBy("payload_hash")
        .agg(F.count("*").alias("hash_count"))
        .filter(F.col("hash_count") > 1)
        .count()
    )

    payload_validation_rows.append({
        "validation_run_id": VALIDATION_RUN_ID,
        "ingestion_batch_id": INGESTION_BATCH_ID,
        "source_name": source_name,
        "table_name": table_name,
        "validation_check": "api_response_success",
        "expected_value": "0 failed responses",
        "actual_value": str(failed_response_count),
        "status": "PASS" if failed_response_count == 0 else "FAIL",
        "issue_detail": None if failed_response_count == 0 else f"{failed_response_count} rows had response_ok != true.",
        "validated_at": datetime.now(timezone.utc).isoformat(),
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })

    payload_validation_rows.append({
        "validation_run_id": VALIDATION_RUN_ID,
        "ingestion_batch_id": INGESTION_BATCH_ID,
        "source_name": source_name,
        "table_name": table_name,
        "validation_check": "raw_payload_not_empty",
        "expected_value": "0 empty payloads",
        "actual_value": str(empty_payload_count),
        "status": "PASS" if empty_payload_count == 0 else "FAIL",
        "issue_detail": None if empty_payload_count == 0 else f"{empty_payload_count} rows had empty raw_payload_json.",
        "validated_at": datetime.now(timezone.utc).isoformat(),
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })

    payload_validation_rows.append({
        "validation_run_id": VALIDATION_RUN_ID,
        "ingestion_batch_id": INGESTION_BATCH_ID,
        "source_name": source_name,
        "table_name": table_name,
        "validation_check": "payload_hash_duplicate_check",
        "expected_value": "0 duplicate payload hashes within source batch",
        "actual_value": str(duplicate_payload_hash_count),
        "status": "PASS" if duplicate_payload_hash_count == 0 else "WARN",
        "issue_detail": None if duplicate_payload_hash_count == 0 else f"{duplicate_payload_hash_count} duplicate payload hash groups found.",
        "validated_at": datetime.now(timezone.utc).isoformat(),
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })


payload_validation_df = spark.createDataFrame(payload_validation_rows, schema=validation_schema)

display(
    payload_validation_df
    .orderBy("status", "source_name", "validation_check")
)

In [0]:
shape_validation_rows = []


def add_shape_check(source_name, table_name, validation_check, expected_value, actual_value, status, issue_detail=None):
    shape_validation_rows.append({
        "validation_run_id": VALIDATION_RUN_ID,
        "ingestion_batch_id": INGESTION_BATCH_ID,
        "source_name": source_name,
        "table_name": table_name,
        "validation_check": validation_check,
        "expected_value": expected_value,
        "actual_value": str(actual_value),
        "status": status,
        "issue_detail": issue_detail,
        "validated_at": datetime.now(timezone.utc).isoformat(),
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })


# GDELT should contain an articles array.
gdelt_df = spark.table(f"{BRONZE_SCHEMA}.bronze_ext_gdelt_doc_raw").filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
gdelt_valid_count = gdelt_df.filter(F.get_json_object("raw_payload_json", "$.articles").isNotNull()).count()
gdelt_total_count = gdelt_df.count()

add_shape_check(
    source_name="gdelt",
    table_name="bronze_ext_gdelt_doc_raw",
    validation_check="payload_shape_articles_array_exists",
    expected_value=f"{gdelt_total_count} rows with $.articles",
    actual_value=gdelt_valid_count,
    status="PASS" if gdelt_valid_count == gdelt_total_count and gdelt_total_count > 0 else "FAIL",
    issue_detail=None if gdelt_valid_count == gdelt_total_count and gdelt_total_count > 0 else "GDELT payload missing expected articles array."
)


# NWS should be GeoJSON FeatureCollection with features.
nws_df = spark.table(f"{BRONZE_SCHEMA}.bronze_ext_nws_alerts_raw").filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
nws_valid_count = (
    nws_df
    .filter(F.get_json_object("raw_payload_json", "$.type") == "FeatureCollection")
    .filter(F.get_json_object("raw_payload_json", "$.features").isNotNull())
    .count()
)
nws_total_count = nws_df.count()

add_shape_check(
    source_name="nws",
    table_name="bronze_ext_nws_alerts_raw",
    validation_check="payload_shape_geojson_feature_collection",
    expected_value=f"{nws_total_count} rows with FeatureCollection and features",
    actual_value=nws_valid_count,
    status="PASS" if nws_valid_count == nws_total_count and nws_total_count > 0 else "FAIL",
    issue_detail=None if nws_valid_count == nws_total_count and nws_total_count > 0 else "NWS payload missing expected GeoJSON FeatureCollection structure."
)


# CISA KEV should contain vulnerabilities.
cisa_df = spark.table(f"{BRONZE_SCHEMA}.bronze_ext_cisa_kev_raw").filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
cisa_valid_count = cisa_df.filter(F.get_json_object("raw_payload_json", "$.vulnerabilities").isNotNull()).count()
cisa_total_count = cisa_df.count()

add_shape_check(
    source_name="cisa",
    table_name="bronze_ext_cisa_kev_raw",
    validation_check="payload_shape_vulnerabilities_array_exists",
    expected_value=f"{cisa_total_count} rows with $.vulnerabilities",
    actual_value=cisa_valid_count,
    status="PASS" if cisa_valid_count == cisa_total_count and cisa_total_count > 0 else "FAIL",
    issue_detail=None if cisa_valid_count == cisa_total_count and cisa_total_count > 0 else "CISA KEV payload missing expected vulnerabilities array."
)


# CPSC usually returns a top-level JSON array of recall records.
cpsc_df = spark.table(f"{BRONZE_SCHEMA}.bronze_ext_cpsc_recalls_raw").filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
cpsc_valid_count = (
    cpsc_df
    .filter(F.substring(F.trim(F.col("raw_payload_json")), 1, 1) == "[")
    .count()
)
cpsc_total_count = cpsc_df.count()

add_shape_check(
    source_name="cpsc",
    table_name="bronze_ext_cpsc_recalls_raw",
    validation_check="payload_shape_top_level_array",
    expected_value=f"{cpsc_total_count} rows with top-level JSON array",
    actual_value=cpsc_valid_count,
    status="PASS" if cpsc_valid_count == cpsc_total_count and cpsc_total_count > 0 else "FAIL",
    issue_detail=None if cpsc_valid_count == cpsc_total_count and cpsc_total_count > 0 else "CPSC recalls payload did not look like a top-level JSON array."
)


# OFAC is stored as raw XML text inside raw_payload_json.
ofac_df = spark.table(f"{BRONZE_SCHEMA}.bronze_ext_ofac_sanctions_raw").filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
ofac_valid_count = (
    ofac_df
    .filter(
        F.lower(F.col("raw_payload_json")).contains("sdn")
        | F.lower(F.col("raw_payload_json")).contains("sanctions")
        | F.lower(F.col("raw_payload_json")).contains("consolidated")
    )
    .count()
)
ofac_total_count = ofac_df.count()

add_shape_check(
    source_name="ofac",
    table_name="bronze_ext_ofac_sanctions_raw",
    validation_check="payload_shape_ofac_xml_text_present",
    expected_value=f"{ofac_total_count} rows with OFAC XML-like sanctions text",
    actual_value=ofac_valid_count,
    status="PASS" if ofac_valid_count == ofac_total_count and ofac_total_count > 0 else "FAIL",
    issue_detail=None if ofac_valid_count == ofac_total_count and ofac_total_count > 0 else "OFAC payload missing expected sanctions XML text."
)


# SEC company submissions should contain cik and filings metadata.
sec_df = spark.table(f"{BRONZE_SCHEMA}.bronze_ext_sec_company_submissions_raw").filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
sec_valid_count = (
    sec_df
    .filter(F.get_json_object("raw_payload_json", "$.cik").isNotNull())
    .filter(F.get_json_object("raw_payload_json", "$.filings").isNotNull())
    .count()
)
sec_total_count = sec_df.count()

add_shape_check(
    source_name="sec_edgar",
    table_name="bronze_ext_sec_company_submissions_raw",
    validation_check="payload_shape_cik_and_filings_exist",
    expected_value=f"{sec_total_count} rows with $.cik and $.filings",
    actual_value=sec_valid_count,
    status="PASS" if sec_valid_count == sec_total_count and sec_total_count > 0 else "FAIL",
    issue_detail=None if sec_valid_count == sec_total_count and sec_total_count > 0 else "SEC payload missing expected cik or filings metadata."
)


# EIA v2 API should contain response data.
eia_df = spark.table(f"{BRONZE_SCHEMA}.bronze_ext_eia_fuel_prices_raw").filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
eia_valid_count = (
    eia_df
    .filter(F.get_json_object("raw_payload_json", "$.response").isNotNull())
    .count()
)
eia_total_count = eia_df.count()

add_shape_check(
    source_name="eia",
    table_name="bronze_ext_eia_fuel_prices_raw",
    validation_check="payload_shape_response_object_exists",
    expected_value=f"{eia_total_count} rows with $.response",
    actual_value=eia_valid_count,
    status="PASS" if eia_valid_count == eia_total_count and eia_total_count > 0 else "FAIL",
    issue_detail=None if eia_valid_count == eia_total_count and eia_total_count > 0 else "EIA payload missing expected response object."
)


shape_validation_df = spark.createDataFrame(shape_validation_rows, schema=validation_schema)

display(
    shape_validation_df
    .orderBy("status", "source_name", "validation_check")
)

In [0]:
all_external_validation_df = (
    table_validation_df
    .unionByName(column_validation_df)
    .unionByName(payload_validation_df)
    .unionByName(shape_validation_df)
)

validation_results_table = f"{BRONZE_SCHEMA}.bronze_ext_validation_results"

# Avoid duplicate rows if this notebook is rerun.
try:
    spark.sql(f"""
    DELETE FROM {validation_results_table}
    WHERE validation_run_id = '{VALIDATION_RUN_ID}'
    """)
    print("Deleted existing validation rows for this validation run.")
except Exception as e:
    print("Validation results table does not exist yet, or no previous rows to delete.")
    print(str(e)[:300])

(
    all_external_validation_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(validation_results_table)
)

print(f"Wrote validation results to {validation_results_table}")

display(
    spark.table(validation_results_table)
    .filter(F.col("validation_run_id") == VALIDATION_RUN_ID)
    .groupBy("status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("status")
)

In [0]:
final_results_df = (
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_validation_results")
    .filter(F.col("validation_run_id") == VALIDATION_RUN_ID)
)

display(
    final_results_df
    .groupBy("status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("status")
)

fail_count = final_results_df.filter(F.col("status") == "FAIL").count()
warn_count = final_results_df.filter(F.col("status") == "WARN").count()
pass_count = final_results_df.filter(F.col("status") == "PASS").count()
total_count = final_results_df.count()

print("Total validation checks:", total_count)
print("PASS:", pass_count)
print("WARN:", warn_count)
print("FAIL:", fail_count)

if fail_count == 0:
    print("✅ Notebook 06 PASSED: External Bronze sources are validated and ready for Silver parsing.")
    print("Next notebook: 07_silver_m5_calendar")
else:
    print("❌ Notebook 06 FAILED: Review failed validation checks before moving to Silver.")